# DeepSeekMath: разобрать GRPO objective

**DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models** · Shao et al. · 2024

[Оригинальная статья](https://arxiv.org/abs/2402.03300)

**Цель урока:** Разделить group advantages, old policy, reference KL и response masks; проверить clipping и постоянные rewards.

## Перед началом

Вводный GRPO, log probabilities, masks и градиенты.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

GRPO использует группу ответов на один prompt для оценки относительного качества без отдельного value model. Важно не смешивать prompts при нормировке rewards. Old policy — распределение, собравшее rollout; reference — политика для регуляризации, если KL включена. Это разные роли.

Reward может быть формальным и проверяемым, но его всё равно можно эксплуатировать. Падающий policy loss не означает улучшение reasoning. Здесь фиксированные искусственные responses; generation, проверка математических ответов и dataset статьи не воспроизводятся.

## Граница воспроизведения

Objective и диагностические инварианты. Не полный GRPO trainer, не обучение DeepSeekMath и не reproduction математических результатов.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`A_i = (r_i - mean(group)) / max(std(group), eps)`. Для reward одинаковых внутри группы advantage=0. Здесь явно выбираем population std (`unbiased=False`); trainer conventions могут отличаться.

`ratio = exp(logpi - logpi_old)`; surrogate=`min(ratio*A, clip(ratio,1-e,1+e)*A)`. Token KL estimator: `exp(logref-logpi) - (logref-logpi) - 1`. Сначала усредняем по валидным токенам каждого ответа, затем по ответам. Old/reference log-probs detached.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Два prompts, по три ответа. Вторая группа имеет одинаковую награду и не даёт относительного сигнала.

In [ ]:
rewards = torch.tensor([[0., 1., 3.], [2., 2., 2.]])
advantage = (rewards - rewards.mean(-1, keepdim=True)) / rewards.std(-1, keepdim=True, unbiased=False).clamp_min(1e-6)
assert torch.isfinite(advantage).all()
assert torch.equal(advantage[1], torch.zeros(3))
assert torch.allclose(advantage.mean(-1), torch.zeros(2), atol=1e-6)
print('advantages:', advantage)
mask = torch.tensor([[[1, 1, 0], [1, 1, 1], [1, 0, 0]], [[1, 1, 0], [1, 1, 1], [1, 0, 0]]], dtype=torch.float)

## Шаг 2

Считаем objective по token log-probs. Это числовой unit experiment: свободные log-probs не являются полноценной нормированной policy.

In [ ]:
old_logp = torch.full((2, 3, 3), -2.)
reference_logp = old_logp.clone()
current_logp = old_logp.clone().requires_grad_()
def objective(current):
    ratio = (current - old_logp).exp()
    A = advantage[..., None]
    surrogate = torch.minimum(ratio * A, ratio.clamp(0.8, 1.2) * A)
    difference = reference_logp - current
    kl = difference.exp() - difference - 1
    token_objective = surrogate - 0.04 * kl
    loss = -((token_objective * mask).sum(-1) / mask.sum(-1).clamp_min(1)).mean()
    return loss, kl
loss, kl = objective(current_logp); loss.backward()
assert torch.equal(kl, torch.zeros_like(kl))
assert torch.equal(current_logp.grad[mask == 0], torch.zeros_like(current_logp.grad[mask == 0]))
assert torch.equal(current_logp.grad[1], torch.zeros_like(current_logp.grad[1]))
print('initial loss:', loss.item(), 'gradient:', current_logp.grad)

## Шаг 3

Визуализируем clipping для положительного и отрицательного advantage, отдельно от KL.

In [ ]:
ratios = torch.linspace(0.4, 1.6, 121)
for A, color in [(1., '#d73535'), (-1., '#777777')]:
    clipped = torch.minimum(ratios * A, ratios.clamp(0.8, 1.2) * A)
    plt.plot(ratios, clipped, label=f'advantage={A}', color=color)
plt.axvline(0.8, ls=':', color='gray'); plt.axvline(1.2, ls=':', color='gray')
plt.xlabel('policy ratio'); plt.ylabel('clipped surrogate'); plt.legend(); plt.show()
varied = old_logp.clone(); varied[0, 0, 2] += 5
assert torch.allclose(objective(varied)[0], objective(old_logp)[0])

## Как читать результат

Для A>0 plateau появляется справа от 1+epsilon; для A<0 — слева от 1-epsilon. Clipping не запрещает любые большие изменения параметров и не гарантирует ограниченный KL. Нулевой advantage не означает отсутствие KL-сигнала, если policy ушла от reference.

## Самостоятельные эксперименты

1. Нормируйте rewards по всему batch и найдите ошибку смысла.
2. Измените current_logp второй группы и проверьте KL gradient.
3. Сравните token-average и response-average при разной длине.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Batch-нормировка сравнивает ответы разных задач. При отличии от reference KL даёт сигнал даже без reward advantage. Token-average сильнее взвешивает длинные ответы; response-average даёт одинаковый вес каждому ответу.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.